# Recaptioning and Data Mixing

**Key insight:** Re-captioning is the single highest-ROI data intervention. Replacing noisy alt-text with structured VLM-generated captions consistently improves model quality more than architecture changes.

**Coverage:**
- VLM-based re-captioning
- Structured caption format
- Caption quality evaluation
- Data mixing strategies
- Training format comparison

**VRAM:** ~4–6 GB for VLM inference (CPU fallback available). **Time:** ~2–3 hours.

**Key papers:**
- Open-Sora 2.0 ([2503.09642](https://arxiv.org/abs/2503.09642))
- HunyuanVideo ([2412.03603](https://arxiv.org/abs/2412.03603))

In [ ]:
import json
import time
import struct
import tarfile
import io
import numpy as np
import torch
from pathlib import Path
from dataclasses import dataclass
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter


## VLM-Based Re-captioning

### The Problem

Web-scraped alt-text is noisy, short, and often completely irrelevant:
- `"IMG_2847.jpg"` — no information at all
- `"click here"` — describes the link, not the image
- `"stock photo"` — describes the source, not the content
- `"beautiful sunset"` — vague, no specifics

Models trained on this corpus learn poor text-image alignment. The model sees thousands of sunsets all labeled "beautiful sunset" with no information about colors, composition, or mood.

### The Solution

Run a Vision-Language Model (BLIP-2, CogVLM, InternVL, LLaVA) over the dataset to generate detailed, accurate captions. This is standard practice in production pipelines — the quality gap between alt-text and VLM captions is large enough that it dominates other engineering decisions.

### Production Considerations

- **Throughput**: billions of images need captioning → batched inference, multiple GPUs, distributed workers
- **Cost**: VLM inference is expensive → use smaller models for first pass, large models for high-value subsets
- **Quality**: VLMs hallucinate → need verification/filtering downstream (covered in notebook 18)
- **Prompting**: the prompt template dramatically affects caption quality and format — this is where most of the engineering effort goes

### Prompt Engineering for Captions

The same VLM with different prompts produces dramatically different outputs. A short prompt gives short captions; a structured prompt gives structured output. The prompt is a hyperparameter that affects your entire training distribution.


In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import random


# ---------------------------------------------------------------------------
# Prompt templates — these are real templates used in production pipelines
# ---------------------------------------------------------------------------

PROMPT_TEMPLATES: dict[str, str] = {
    "short": "Describe this image briefly.",
    "detailed": (
        "Describe this image in detail, including objects, actions, "
        "colors, composition, and mood."
    ),
    "structured": (
        "Describe this image using the following structure:\n"
        "1) Main subject\n"
        "2) Setting / background\n"
        "3) Actions or events\n"
        "4) Visual style and color palette\n"
        "5) Camera angle and framing"
    ),
    "cinematographic": (
        "You are a professional cinematographer. Describe this image as a "
        "shot description for a film director, covering: composition, lighting, "
        "depth of field, color grading, mood, and narrative."
    ),
}


# ---------------------------------------------------------------------------
# Mock structured captions — realistic examples for demonstration
# ---------------------------------------------------------------------------

MOCK_CAPTIONS: dict[str, list[str]] = {
    "short": [
        "A golden retriever playing on a beach at sunset.",
        "City street at night with neon signs reflecting on wet pavement.",
        "A woman reading a book in a sunlit library.",
    ],
    "detailed": [
        (
            "A golden retriever with a shiny, wavy coat is bounding through shallow "
            "ocean waves on a sandy beach. The sky is painted in deep oranges and "
            "pinks as the sun sets on the horizon. The dog's mouth is open in a "
            "joyful expression, ears flying backward. The wet sand reflects the "
            "warm sunset colors. The mood is carefree and energetic."
        ),
        (
            "A rain-slicked urban street at night, lit by a dense array of neon "
            "signs in red, blue, and yellow from storefronts and restaurants. "
            "The reflections stretch in distorted patterns across the wet asphalt. "
            "A few blurred pedestrian figures move in the background. "
            "The atmosphere is atmospheric and slightly melancholic."
        ),
        (
            "A young woman with dark curly hair sits in a leather armchair in a "
            "high-ceilinged library, surrounded by floor-to-ceiling wooden bookshelves. "
            "Dust motes float in a shaft of warm afternoon sunlight streaming through "
            "a tall arched window. She holds a hardcover book open in both hands, "
            "eyes focused downward. The overall palette is warm amber and deep brown."
        ),
    ],
    "structured": [
        (
            "1) Main subject: A golden retriever dog, adult, with a thick wavy golden coat.\n"
            "2) Setting: Sandy ocean beach with shallow waves, coastal landscape.\n"
            "3) Actions: Dog is running and jumping through the surf, mid-leap.\n"
            "4) Visual style: Warm golden-hour light, saturated oranges and pinks, high contrast.\n"
            "5) Camera angle: Low angle, wide shot, dog positioned in center-left frame."
        ),
        (
            "1) Main subject: Urban street scene, no clear central figure.\n"
            "2) Setting: City street at night, commercial district, wet pavement.\n"
            "3) Actions: Pedestrians walking in background, no primary action.\n"
            "4) Visual style: Neon-lit noir aesthetic, vivid reds and blues, high contrast, wet reflections.\n"
            "5) Camera angle: Eye-level, slight low angle, looking down the street."
        ),
        (
            "1) Main subject: Young woman, approximately 25-30 years old, dark curly hair.\n"
            "2) Setting: Classic library interior with tall wooden bookshelves.\n"
            "3) Actions: Reading a hardcover book, seated, relaxed posture.\n"
            "4) Visual style: Warm amber tones, soft diffused sunlight, dust particles visible.\n"
            "5) Camera angle: Medium shot, slightly above eye level, subject in center frame."
        ),
    ],
}


class CaptionGenerator:
    """Wraps a VLM for batch captioning with configurable prompt templates.

    Attempts to load a real BLIP model; falls back to structured mock output
    if transformers is unavailable or device has insufficient VRAM.
    """

    def __init__(self, device: Optional[str] = None, use_mock: bool = False):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.processor = None
        self.use_mock = use_mock

        if not use_mock:
            self._try_load_blip()

    def _try_load_blip(self) -> None:
        """Attempt to load BLIP-2; set use_mock if unavailable."""
        try:
            from transformers import BlipProcessor, BlipForConditionalGeneration
            print("Loading BLIP (Salesforce/blip-image-captioning-base)...")
            self.processor = BlipProcessor.from_pretrained(
                "Salesforce/blip-image-captioning-base"
            )
            self.model = BlipForConditionalGeneration.from_pretrained(
                "Salesforce/blip-image-captioning-base"
            ).to(self.device)
            self.model.eval()
            print(f"BLIP loaded on {self.device}")
        except Exception as e:
            print(f"Could not load BLIP ({e}). Using mock captions.")
            self.use_mock = True

    def _mock_caption(self, prompt_template: str, idx: int) -> str:
        """Return a realistic mock caption matched to prompt style."""
        # Determine which style bucket to use
        if "briefly" in prompt_template:
            style = "short"
        elif "structure" in prompt_template or "1)" in prompt_template:
            style = "structured"
        else:
            style = "detailed"
        options = MOCK_CAPTIONS[style]
        return options[idx % len(options)]

    def _real_caption(self, image: Image.Image, prompt_template: str) -> str:
        """Generate caption using the loaded BLIP model."""
        assert self.processor is not None and self.model is not None
        inputs = self.processor(
            images=image,
            text=prompt_template,
            return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=200)
        return self.processor.decode(out[0], skip_special_tokens=True)

    def batch_caption(
        self,
        images: list[Image.Image],
        prompt_template: str = PROMPT_TEMPLATES["detailed"],
    ) -> list[str]:
        """Caption a batch of images with the given prompt template.

        Returns:
            List of caption strings, one per image.
        """
        captions = []
        for idx, img in enumerate(images):
            if self.use_mock:
                captions.append(self._mock_caption(prompt_template, idx))
            else:
                captions.append(self._real_caption(img, prompt_template))
        return captions


# ---------------------------------------------------------------------------
# Demo: compare prompt templates side by side
# ---------------------------------------------------------------------------

def make_synthetic_images(n: int = 3) -> list[Image.Image]:
    """Generate simple colored synthetic images for demo purposes."""
    rng = np.random.default_rng(42)
    images = []
    for _ in range(n):
        arr = rng.integers(0, 255, (128, 128, 3), dtype=np.uint8)
        images.append(Image.fromarray(arr))
    return images


def benchmark_throughput(
    generator: CaptionGenerator,
    images: list[Image.Image],
    prompt_template: str,
    n_runs: int = 3,
) -> float:
    """Measure caption throughput in images/second."""
    t0 = time.perf_counter()
    for _ in range(n_runs):
        generator.batch_caption(images, prompt_template)
    elapsed = time.perf_counter() - t0
    total_images = len(images) * n_runs
    return total_images / elapsed


# Run demo
gen = CaptionGenerator(use_mock=True)  # set use_mock=False to use real BLIP
synth_images = make_synthetic_images(3)

print("=" * 70)
print("PROMPT TEMPLATE COMPARISON (3 synthetic images)")
print("=" * 70)

for template_name, template_text in PROMPT_TEMPLATES.items():
    captions = gen.batch_caption(synth_images, template_text)
    print(f"\n--- Template: '{template_name}' ---")
    print(f"Prompt: {template_text[:80]}{'...' if len(template_text) > 80 else ''}")
    for i, cap in enumerate(captions):
        preview = cap[:120].replace("\n", " | ")
        print(f"  Image {i+1}: {preview}{'...' if len(cap) > 120 else ''}")

# Throughput benchmark
print("\n" + "=" * 70)
print("THROUGHPUT BENCHMARK")
print("=" * 70)
for template_name, template_text in list(PROMPT_TEMPLATES.items())[:2]:
    tput = benchmark_throughput(gen, synth_images, template_text)
    print(f"  '{template_name}' prompt: {tput:.1f} images/sec (mock)")
print("\nNote: real VLM throughput on RTX 2080: ~10-40 images/sec depending on model size.")


## Structured Caption Format

### Why Structure Matters

Unstructured captions vary wildly in what they cover. One caption might focus on colors, another on composition, another on emotional tone. This inconsistency propagates into training: the model can't reliably learn to control attributes that are only sometimes mentioned.

Structured captions ensure **consistent coverage** of key attributes across the entire dataset. This pays dividends at inference time — you can condition on specific attributes with high reliability.

### Standard Fields

| Field | Description | Example |
|-------|-------------|--------|
| **scene** | Overall scene description | `"coastal beach at golden hour"` |
| **subjects** | Main objects/people and their attributes | `["adult golden retriever", "wet sand"]` |
| **actions** | What's happening | `["dog running through surf", "waves breaking"]` |
| **style** | Visual style, lighting, color palette | `"warm saturated, high contrast, filmic"` |
| **camera** | Angle, distance, movement (critical for video) | `"low angle wide shot, static"` |
| **text_in_image** | Any visible text content | `"SALE 50% OFF"` or `null` |

### Conversion to Training Text

Structured captions are stored as structured data but converted to natural language at training time. Different verbosity levels allow you to train models that respond to both short and detailed prompts.


In [ ]:
import re
from typing import Literal


@dataclass
class StructuredCaption:
    """Structured representation of an image caption.

    All fields except scene are optional to allow partial parsing from
    VLM output that doesn't cover every category.
    """

    scene: str
    subjects: list[str] = field(default_factory=list)
    actions: list[str] = field(default_factory=list)
    style: str = ""
    camera: str = ""
    text_in_image: Optional[str] = None

    def field_coverage(self) -> float:
        """Fraction of optional fields that are non-empty (0.0 to 1.0)."""
        optional_fields = [self.subjects, self.actions, self.style, self.camera]
        filled = sum(1 for f in optional_fields if f)
        return filled / len(optional_fields)


def parse_structured_caption(raw: str) -> StructuredCaption:
    """Parse VLM output into StructuredCaption using numbered-list heuristics.

    Handles both structured (numbered) and unstructured caption formats.
    Falls back gracefully: if parsing fails, everything goes into 'scene'.
    """
    # Try to parse numbered structure: "1) ...", "2) ..."
    numbered = re.findall(r"\d+\)\s*([^\n]+(?:\n(?!\d+\)).*)*)", raw.strip())

    if len(numbered) >= 3:
        # Structured format detected
        def clean(s: str) -> str:
            return s.strip().replace("\n", " ")

        # Field 1 → subjects ("Main subject: ...") or direct text
        subj_raw = re.sub(r"^[Mm]ain\s+subject\s*:\s*", "", clean(numbered[0]))
        # Split comma-separated subjects
        subjects = [s.strip() for s in subj_raw.split(",") if s.strip()]

        # Field 2 → scene
        scene_raw = re.sub(r"^[Ss]etting\s*[/\\]?\s*[Bb]ackground\s*:\s*", "", clean(numbered[1]))

        # Field 3 → actions
        action_raw = re.sub(r"^[Aa]ctions?\s*(?:or events?)?\s*:\s*", "", clean(numbered[2]))
        actions = [a.strip() for a in re.split(r",\s*|;\s*", action_raw) if a.strip()]

        # Field 4 → style
        style = ""
        if len(numbered) >= 4:
            style = re.sub(r"^[Vv]isual\s+style\s*[^:]*:\s*", "", clean(numbered[3]))

        # Field 5 → camera
        camera = ""
        if len(numbered) >= 5:
            camera = re.sub(r"^[Cc]amera\s*[^:]*:\s*", "", clean(numbered[4]))

        return StructuredCaption(
            scene=scene_raw,
            subjects=subjects,
            actions=actions,
            style=style,
            camera=camera,
        )

    # Unstructured: put everything in scene, attempt action extraction
    sentences = re.split(r"(?<=[.!?])\s+", raw.strip())
    scene = sentences[0] if sentences else raw.strip()
    # Heuristic: sentences with action verbs → actions
    action_verbs = {"running", "walking", "sitting", "reading", "playing", "jumping",
                    "standing", "flying", "driving", "swimming", "holding", "wearing"}
    actions = []
    for sent in sentences[1:]:
        words = set(sent.lower().split())
        if words & action_verbs:
            actions.append(sent.strip())

    return StructuredCaption(
        scene=scene,
        actions=actions,
    )


VerbosityLevel = Literal["short", "medium", "detailed"]


def caption_to_training_text(
    caption: StructuredCaption,
    verbosity: VerbosityLevel = "medium",
) -> str:
    """Convert StructuredCaption to natural language for training.

    Args:
        caption: Structured caption object.
        verbosity: Controls how much detail to include.

    Returns:
        Natural language caption string.
    """
    if verbosity == "short":
        parts = [caption.scene]
        if caption.subjects:
            parts[0] = f"{caption.subjects[0]}, {caption.scene}"
        return " ".join(parts)

    elif verbosity == "medium":
        parts = [caption.scene]
        if caption.actions:
            parts.append(caption.actions[0])
        if caption.style:
            parts.append(f"Style: {caption.style}.")
        return " ".join(parts)

    else:  # detailed
        parts = [caption.scene]
        if caption.subjects:
            parts.append(f"Subjects: {', '.join(caption.subjects)}")
        if caption.actions:
            parts.append(f"Actions: {', '.join(caption.actions)}")
        if caption.style:
            parts.append(f"Visual style: {caption.style}")
        if caption.camera:
            parts.append(f"Camera: {caption.camera}")
        if caption.text_in_image:
            parts.append(f"Text visible: '{caption.text_in_image}'")
        return ". ".join(parts) + "."


# ---------------------------------------------------------------------------
# Caption Quality Scorer
# ---------------------------------------------------------------------------

VAGUE_WORDS = {
    "beautiful", "nice", "good", "great", "amazing", "wonderful", "lovely",
    "photo", "image", "picture", "stock", "various", "different", "some",
    "many", "things", "stuff", "misc", "assorted",
}

CONCRETE_NOUNS = {
    "dog", "cat", "person", "woman", "man", "child", "car", "tree", "building",
    "mountain", "ocean", "river", "book", "chair", "table", "window", "sky",
    "cloud", "flower", "bird", "horse", "bicycle", "camera", "phone", "laptop",
    "restaurant", "street", "beach", "forest", "library", "kitchen", "bedroom",
}


@dataclass
class QualityScore:
    """Quality breakdown for a single caption."""

    length_score: float       # 0-1, optimal 50-200 words
    specificity_score: float  # 0-1, ratio of concrete to vague words
    coverage_score: float     # 0-1, fraction of structured fields filled
    repetition_score: float   # 0-1, penalize repeated n-grams

    @property
    def total(self) -> float:
        """Weighted average of component scores."""
        weights = [0.2, 0.35, 0.35, 0.1]
        scores = [self.length_score, self.specificity_score,
                  self.coverage_score, self.repetition_score]
        return sum(w * s for w, s in zip(weights, scores))


class CaptionQualityScorer:
    """Score captions on length, specificity, field coverage, and repetition."""

    def score(self, raw: str, structured: Optional[StructuredCaption] = None) -> QualityScore:
        """Compute quality scores for a caption string."""
        words = raw.lower().split()
        n_words = len(words)

        # Length score: optimal range 50-200 words
        if n_words < 5:
            length_score = 0.0
        elif n_words < 20:
            length_score = n_words / 20 * 0.5
        elif n_words <= 200:
            length_score = 0.5 + (n_words - 20) / 180 * 0.5
        else:
            # Penalize excessively long captions
            length_score = max(0.0, 1.0 - (n_words - 200) / 300)

        # Specificity: concrete nouns vs vague words
        word_set = set(words)
        n_concrete = len(word_set & CONCRETE_NOUNS)
        n_vague = len(word_set & VAGUE_WORDS)
        if n_words == 0:
            specificity_score = 0.0
        else:
            specificity_score = min(1.0, (n_concrete * 2) / max(1, n_words / 10))
            specificity_score *= max(0.3, 1.0 - n_vague * 0.15)

        # Coverage: structured fields
        coverage_score = structured.field_coverage() if structured else 0.2

        # Repetition: bigram overlap
        if n_words < 4:
            repetition_score = 1.0
        else:
            bigrams = [(words[i], words[i + 1]) for i in range(len(words) - 1)]
            counts = Counter(bigrams)
            repeated = sum(1 for c in counts.values() if c > 1)
            repetition_score = max(0.0, 1.0 - repeated / max(1, len(bigrams)) * 3)

        return QualityScore(
            length_score=round(length_score, 3),
            specificity_score=round(specificity_score, 3),
            coverage_score=round(coverage_score, 3),
            repetition_score=round(repetition_score, 3),
        )


# ---------------------------------------------------------------------------
# Demo: score a realistic caption set
# ---------------------------------------------------------------------------

test_captions: list[tuple[str, str]] = [
    ("IMG_2847.jpg", "IMG_2847.jpg"),
    ("click here", "click here"),
    ("beautiful sunset", "beautiful sunset"),
    ("golden hour beach", "A photo at the beach during golden hour."),
    ("unstructured (medium)", MOCK_CAPTIONS["detailed"][0]),
    ("structured (full)", MOCK_CAPTIONS["structured"][0]),
]

scorer = CaptionQualityScorer()

print(f"{'Caption Label':<22} {'Length':>7} {'Specific':>9} {'Coverage':>9} {'Repeat':>7} {'TOTAL':>7}")
print("-" * 65)

all_scores: list[float] = []
for label, raw in test_captions:
    structured = parse_structured_caption(raw)
    q = scorer.score(raw, structured)
    all_scores.append(q.total)
    print(
        f"{label:<22} {q.length_score:>7.3f} {q.specificity_score:>9.3f} "
        f"{q.coverage_score:>9.3f} {q.repetition_score:>7.3f} {q.total:>7.3f}"
    )

threshold = 0.25
low_quality = [(lbl, s) for (lbl, _), s in zip(test_captions, all_scores) if s < threshold]
print(f"\nLow quality (score < {threshold}): {[lbl for lbl, _ in low_quality]}")
print("These would be flagged for re-captioning.")

# Verbosity demo
print("\n--- Verbosity conversion demo ---")
sample_structured = parse_structured_caption(MOCK_CAPTIONS["structured"][0])
for v in ["short", "medium", "detailed"]:
    out = caption_to_training_text(sample_structured, verbosity=v)  # type: ignore[arg-type]
    print(f"  [{v}] {out[:100]}{'...' if len(out) > 100 else ''}")


## Data Mixing Strategies

### The Problem

Training data is rarely homogeneous. A real dataset might include:
- 500M web-scraped images (noisy, diverse)
- 10M curated aesthetic images (high quality, narrow distribution)
- 2M licensed commercial images (high quality, specific domains)
- 50M synthetic images (perfect captions, distribution mismatch)

Naively concatenating these gives you a distribution dominated by the web-scraped images. Your model will be mediocre at everything rather than excellent at what matters.

### Key Concepts

**Source weighting:** Upweight high-quality curated data relative to noisy web data. A 10M curated set can be worth more than 500M web images if weighted correctly.

**Domain mixing:** Balance across content types (nature, people, objects, text, abstract art). Without explicit domain balancing, common web content (selfies, food photos) will dominate.

**Curriculum learning:** Start training with easy/clean data, progressively introduce harder/noisier data. The model first learns clean patterns, then generalizes. This is used in Open-Sora 2.0 and several other large video models.

**Temperature sampling:** Controls how much the distribution is flattened across sources. Temperature `T=1.0` preserves original ratios; `T → 0` concentrates all weight on the highest-quality source; `T → ∞` approaches uniform.

$$p_i \propto w_i^{1/T}$$

### Common Failure Modes

- **Over-representation of common scenes**: the model learns to generate sunsets and food extremely well, fails at rare content
- **Under-representation of rare concepts**: text rendering, hands, unusual camera angles all tend to be underrepresented and require explicit upweighting
- **Domain collapse**: if one source dominates, the model loses diversity and mode-collapses toward that source's aesthetic
- **Quality dilution**: too much noisy data overwhelms quality signal even with upweighting — there's a ratio below which quality data stops helping


In [ ]:
from dataclasses import dataclass as dc


@dataclass
class DataSource:
    """Metadata for a single data source in a mixed dataset."""

    name: str
    size: int           # number of samples
    quality_score: float  # 0.0 (noisy) to 1.0 (curated)
    domain: str = "general"


class DataMixer:
    """Compute sampling weights and sample batches across multiple data sources."""

    STRATEGIES = ("proportional", "sqrt", "quality", "equal")

    def __init__(self, sources: dict[str, DataSource]) -> None:
        self.sources = sources
        self._names = list(sources.keys())
        self._rng = np.random.default_rng(0)

    def compute_weights(
        self,
        strategy: str,
        temperature: float = 1.0,
    ) -> dict[str, float]:
        """Compute normalized sampling weights for each source.

        Args:
            strategy: One of 'proportional', 'sqrt', 'quality', 'equal'.
            temperature: Sampling temperature (1.0 = unchanged, <1 = sharpen, >1 = flatten).

        Returns:
            Dict mapping source name to normalized weight.
        """
        assert strategy in self.STRATEGIES, f"Unknown strategy: {strategy}"

        raw: dict[str, float] = {}
        if strategy == "proportional":
            raw = {k: float(s.size) for k, s in self.sources.items()}
        elif strategy == "sqrt":
            raw = {k: float(s.size) ** 0.5 for k, s in self.sources.items()}
        elif strategy == "quality":
            raw = {k: s.quality_score for k, s in self.sources.items()}
        elif strategy == "equal":
            raw = {k: 1.0 for k in self.sources}

        # Apply temperature
        if temperature != 1.0:
            raw = {k: v ** (1.0 / temperature) for k, v in raw.items()}

        total = sum(raw.values())
        return {k: v / total for k, v in raw.items()}

    def sample_batch(
        self,
        batch_size: int,
        strategy: str = "sqrt",
        temperature: float = 1.0,
    ) -> list[tuple[str, int]]:
        """Sample (source_name, index) pairs for a training batch.

        Returns:
            List of (source_name, sample_index) tuples.
        """
        weights = self.compute_weights(strategy, temperature)
        names = list(weights.keys())
        probs = np.array([weights[n] for n in names])
        chosen = self._rng.choice(len(names), size=batch_size, p=probs)
        result = []
        for src_idx in chosen:
            src_name = names[src_idx]
            sample_idx = int(self._rng.integers(0, self.sources[src_name].size))
            result.append((src_name, sample_idx))
        return result

    def curriculum_schedule(
        self,
        epoch: int,
        total_epochs: int,
    ) -> dict[str, float]:
        """Compute epoch-specific weights shifting from clean to mixed data.

        Early epochs: high weight on quality sources.
        Late epochs: proportional mixing. Linear interpolation between.

        Args:
            epoch: Current epoch (0-indexed).
            total_epochs: Total training epochs.

        Returns:
            Normalized weight dict for this epoch.
        """
        t = epoch / max(1, total_epochs - 1)  # 0.0 → 1.0

        quality_weights = self.compute_weights("quality")
        prop_weights = self.compute_weights("proportional")

        # Interpolate: start at quality, end at proportional
        blended = {
            k: (1 - t) * quality_weights[k] + t * prop_weights[k]
            for k in self._names
        }
        total = sum(blended.values())
        return {k: v / total for k, v in blended.items()}


# ---------------------------------------------------------------------------
# Demo setup
# ---------------------------------------------------------------------------

sources = {
    "web_scraped":   DataSource("web_scraped",   size=500_000_000, quality_score=0.2, domain="general"),
    "aesthetic":     DataSource("aesthetic",     size=10_000_000,  quality_score=0.85, domain="art"),
    "commercial":    DataSource("commercial",    size=2_000_000,   quality_score=0.90, domain="product"),
    "synthetic":     DataSource("synthetic",     size=50_000_000,  quality_score=0.75, domain="general"),
}

mixer = DataMixer(sources)

# Strategy comparison
print("=" * 65)
print("SAMPLING WEIGHT COMPARISON ACROSS STRATEGIES")
print("=" * 65)
print(f"{'Source':<15} {'Size':>12} {'Q':>5} | ", end="")
for strat in DataMixer.STRATEGIES:
    print(f"{strat:>13}", end="")
print()
print("-" * 90)

all_weights = {strat: mixer.compute_weights(strat) for strat in DataMixer.STRATEGIES}

for name, src in sources.items():
    print(f"{name:<15} {src.size:>12,} {src.quality_score:>5.2f} | ", end="")
    for strat in DataMixer.STRATEGIES:
        w = all_weights[strat][name]
        print(f"{w:>12.1%} ", end="")
    print()

# Batch sample demo
print("\n--- Batch sample (sqrt strategy, batch_size=16) ---")
batch = mixer.sample_batch(16, strategy="sqrt")
batch_counts = Counter(src for src, _ in batch)
for name in sources:
    print(f"  {name}: {batch_counts.get(name, 0)} samples")

# Curriculum schedule visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: pie charts for each strategy
ax_strategies = axes[0]
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
n_strategies = len(DataMixer.STRATEGIES)
fig2, axes2 = plt.subplots(1, n_strategies, figsize=(16, 4))
for ax, strat in zip(axes2, DataMixer.STRATEGIES):
    weights = all_weights[strat]
    labels = list(weights.keys())
    vals = list(weights.values())
    wedges, texts, autotexts = ax.pie(
        vals,
        labels=[l.replace("_", "\n") for l in labels],
        colors=colors,
        autopct="%1.1f%%",
        startangle=90,
        textprops={"fontsize": 8},
    )
    ax.set_title(f"'{strat}'\nstrategy", fontsize=10)
plt.suptitle("Sampling Weight Distribution by Strategy", fontsize=12)
plt.tight_layout()
plt.show()

# Right: curriculum schedule over epochs
ax = axes[1]
epochs = range(0, 20)
for name, color in zip(sources.keys(), colors):
    weights_over_time = [
        mixer.curriculum_schedule(e, 20)[name] for e in epochs
    ]
    ax.plot(epochs, weights_over_time, label=name.replace("_", " "), color=color, linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Sampling Weight")
ax.set_title("Curriculum Schedule\n(quality → proportional over training)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
axes[0].axis("off")  # placeholder for spacing
axes[0].text(0.5, 0.5, "See pie charts above", ha="center", va="center", fontsize=10)
plt.tight_layout()
plt.show()

print("\nCurriculum schedule at key epochs:")
for epoch in [0, 5, 10, 19]:
    sched = mixer.curriculum_schedule(epoch, 20)
    parts = ", ".join(f"{k}: {v:.1%}" for k, v in sched.items())
    print(f"  Epoch {epoch:>2}: {parts}")


## Training Data Formats

Format choice has a significant impact on training throughput and engineering complexity. The wrong format can become a bottleneck before you ever hit GPU utilization limits.

| Format | Streaming | Random Access | Resumable | Shuffling | Ecosystem |
|--------|-----------|---------------|-----------|-----------|----------|
| **WebDataset** | Yes (tar) | No | With sharding | Shard-level | Oldest, widest support |
| **MDS (Mosaic)** | Yes | Yes | Yes | Built-in | Good with Composer |
| **LitData** | Yes | Yes | Yes | Built-in | Lightning ecosystem |
| **Lance** | Yes | Yes (100x Parquet) | Yes | Built-in + ANN | Video-native, vector search |

### When to Use Each

**WebDataset**: When you need maximum ecosystem compatibility, are streaming from cloud storage, and can live without random access. Tar files are universally readable. The shuffle granularity (per-shard) is a real limitation — requires aggressive shard-level shuffling.

**MDS**: Good general-purpose choice when you need random access for weighted sampling and are already in the Composer/MosaicML ecosystem.

**LitData**: Strong choice in Lightning ecosystem. Optimized for distributed training with automatic data partitioning.

**Lance**: Best for video data and when you need vector similarity search (e.g., deduplication by embedding). 100x faster random access than Parquet makes it practical for large-scale training. Covered in depth in notebook 09.

Reference: notebook 09 for Lance deep dive.


In [ ]:
import tempfile
import os


# ---------------------------------------------------------------------------
# WebDataset: write and read tar shards
# ---------------------------------------------------------------------------

def make_fake_image_bytes(width: int = 64, height: int = 64) -> bytes:
    """Generate a minimal valid PNG as bytes for demo."""
    img = Image.fromarray(
        np.random.default_rng(42).integers(0, 255, (height, width, 3), dtype=np.uint8)
    )
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def write_webdataset_shard(
    path: Path,
    samples: list[dict[str, bytes | str]],
) -> None:
    """Write samples to a WebDataset-compatible tar shard.

    Each sample should have keys like 'png', 'txt', 'json'.
    """
    with tarfile.open(path, "w") as tar:
        for idx, sample in enumerate(samples):
            key_prefix = f"{idx:06d}"
            for ext, data in sample.items():
                raw = data.encode() if isinstance(data, str) else data
                info = tarfile.TarInfo(name=f"{key_prefix}.{ext}")
                info.size = len(raw)
                tar.addfile(info, io.BytesIO(raw))


def read_webdataset_shard(path: Path) -> list[dict[str, bytes]]:
    """Read all samples from a WebDataset tar shard.

    Groups files by their numeric prefix, returns list of sample dicts.
    """
    groups: dict[str, dict[str, bytes]] = {}
    with tarfile.open(path, "r") as tar:
        for member in tar.getmembers():
            name = member.name
            parts = name.rsplit(".", 1)
            if len(parts) != 2:
                continue
            key, ext = parts
            f = tar.extractfile(member)
            if f is not None:
                groups.setdefault(key, {})[ext] = f.read()
    return list(groups.values())


# ---------------------------------------------------------------------------
# Mock MDS: demonstrate index-based random access concept
# ---------------------------------------------------------------------------

class MockMDSWriter:
    """Minimal mock of MDS format: index file + data file.

    Real MDS uses a more sophisticated binary format with compression.
    This illustrates the core concept: an index enables O(1) random access.
    """

    def __init__(self, out_dir: Path) -> None:
        self.out_dir = out_dir
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.data_path = out_dir / "data.bin"
        self.index_path = out_dir / "index.json"
        self._offsets: list[tuple[int, int]] = []  # (offset, length)
        self._data_file = open(self.data_path, "wb")

    def write(self, sample: dict) -> None:
        """Write a sample; record its byte offset in the index."""
        raw = json.dumps(sample).encode()
        offset = self._data_file.tell()
        self._data_file.write(raw)
        self._offsets.append((offset, len(raw)))

    def finalize(self) -> None:
        """Flush data and write the index."""
        self._data_file.close()
        with open(self.index_path, "w") as f:
            json.dump({"offsets": self._offsets, "n_samples": len(self._offsets)}, f)


class MockMDSReader:
    """Read from mock MDS dataset with O(1) random access."""

    def __init__(self, data_dir: Path) -> None:
        with open(data_dir / "index.json") as f:
            meta = json.load(f)
        self._offsets: list[tuple[int, int]] = [tuple(x) for x in meta["offsets"]]  # type: ignore
        self.n_samples: int = meta["n_samples"]
        self._data_path = data_dir / "data.bin"

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> dict:
        """O(1) random access via index."""
        offset, length = self._offsets[idx]
        with open(self._data_path, "rb") as f:
            f.seek(offset)
            return json.loads(f.read(length))


# ---------------------------------------------------------------------------
# Format recommendation function
# ---------------------------------------------------------------------------

def recommend_format(
    dataset_size: int,
    needs_random_access: bool,
    needs_video: bool,
    needs_vector_search: bool,
) -> str:
    """Recommend a training data format based on requirements.

    Args:
        dataset_size: Number of samples.
        needs_random_access: Whether weighted/non-sequential sampling is required.
        needs_video: Whether samples contain video data.
        needs_vector_search: Whether ANN vector search is needed (e.g., dedup).

    Returns:
        String with format recommendation and reasoning.
    """
    reasons: list[str] = []

    if needs_vector_search:
        reasons.append("Lance: native ANN vector search + fast random access")
        return "LANCE — " + reasons[0]

    if needs_video and needs_random_access:
        reasons.append("Lance: video-native columnar format, 100x Parquet random access")
        return "LANCE — " + reasons[0]

    if needs_random_access and dataset_size > 10_000_000:
        reasons.append("MDS: built-in random access + shuffling, scales to billions of samples")
        return "MDS — " + reasons[0]

    if needs_random_access:
        reasons.append("MDS or LitData: both support random access; choose based on training framework")
        return "MDS/LitData — " + reasons[0]

    if dataset_size > 100_000_000:
        reasons.append("WebDataset: streaming from cloud, universal tooling, minimal overhead")
        return "WebDataset — " + reasons[0]

    reasons.append("WebDataset: simplest format for sequential streaming pipelines")
    return "WebDataset — " + reasons[0]


# ---------------------------------------------------------------------------
# Run demos
# ---------------------------------------------------------------------------

with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)

    # --- WebDataset demo ---
    print("=" * 60)
    print("WEBDATASET TAR SHARD DEMO")
    print("=" * 60)

    n_samples = 20
    samples = [
        {
            "png": make_fake_image_bytes(),
            "txt": f"A synthetic test image number {i}.",
            "json": json.dumps({"id": i, "quality": 0.8}),
        }
        for i in range(n_samples)
    ]

    shard_path = tmp / "shard-000000.tar"

    t0 = time.perf_counter()
    write_webdataset_shard(shard_path, samples)
    write_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    read_back = read_webdataset_shard(shard_path)
    read_time = time.perf_counter() - t0

    shard_size_kb = shard_path.stat().st_size / 1024
    print(f"  Written {n_samples} samples to {shard_path.name}")
    print(f"  Shard size: {shard_size_kb:.1f} KB")
    print(f"  Write time: {write_time*1000:.1f} ms")
    print(f"  Sequential read time: {read_time*1000:.1f} ms ({n_samples/read_time:.0f} samples/sec)")
    print(f"  Sample keys: {list(read_back[0].keys())}")
    print(f"  Limitation: no random access — must read sequentially from position 0")

    # --- Mock MDS demo ---
    print("\n" + "=" * 60)
    print("MOCK MDS RANDOM ACCESS DEMO")
    print("=" * 60)

    mds_dir = tmp / "mds_dataset"
    writer = MockMDSWriter(mds_dir)
    for i in range(n_samples):
        writer.write({"id": i, "caption": f"Sample {i}", "quality": round(0.5 + i / n_samples * 0.5, 2)})
    writer.finalize()

    reader = MockMDSReader(mds_dir)
    print(f"  Dataset size: {len(reader)} samples")

    # Demonstrate O(1) random access — jump to last, first, middle
    for idx in [0, 10, 19]:
        sample = reader[idx]
        print(f"  reader[{idx}] → {sample}")

    # Sequential throughput
    t0 = time.perf_counter()
    _ = [reader[i] for i in range(len(reader))]
    seq_time = time.perf_counter() - t0
    print(f"  Sequential read: {n_samples/seq_time:.0f} samples/sec")

    # Random access throughput
    indices = np.random.default_rng(0).integers(0, n_samples, size=n_samples)
    t0 = time.perf_counter()
    _ = [reader[int(i)] for i in indices]
    rand_time = time.perf_counter() - t0
    print(f"  Random access: {n_samples/rand_time:.0f} samples/sec")

    # --- Format recommendation ---
    print("\n" + "=" * 60)
    print("FORMAT RECOMMENDATION GUIDE")
    print("=" * 60)

    scenarios = [
        ("500M images, streaming only",       500_000_000, False, False, False),
        ("50M images, weighted sampling",      50_000_000,  True,  False, False),
        ("10M video clips, random access",     10_000_000,  True,  True,  False),
        ("100M images + embedding dedup",     100_000_000,  True,  False, True),
        ("1M images, small experiment",         1_000_000,  False, False, False),
    ]
    for label, size, rand, vid, vec in scenarios:
        rec = recommend_format(size, rand, vid, vec)
        print(f"  {label:<35} → {rec}")


## Foundation Model Auto-Labeling

Re-captioning is one piece of a broader **auto-labeling** pipeline. Modern data teams use a stack of foundation models to extract rich annotations without manual labeling:

### The Auto-Labeling Stack

| Model | Task | Output |
|---|---|---|
| **SAM** (Segment Anything) | Class-agnostic segmentation | Binary masks for every object — no class labels needed |
| **CLIP** | Image-text alignment | Zero-shot classification: "is this a car?" via cosine similarity |
| **Grounding DINO** | Text-grounded detection | Bounding boxes from natural language: "find all pedestrians" |
| **VLM** (Qwen-VL, InternVL) | Structured metadata extraction | JSON output: scene type, camera motion, lighting, objects, actions |

### The Consensus Pattern

No single model is reliable enough alone. Production auto-labeling uses **consensus scoring**:

1. Run 2-3 models on the same input (e.g., CLIP classification + VLM description + Grounding DINO detection)
2. If models **agree** → high-confidence label
3. If models **disagree** → flag for review or discard
4. For critical labels: run the same VLM with **multiple prompts** and check consistency

```python
# Conceptual consensus labeling
def consensus_label(frame, query: str) -> dict:
    """Label a frame using multiple foundation models with consensus."""
    # CLIP: zero-shot classification
    clip_score = clip_similarity(frame, query)  # e.g., 0.31
    
    # Grounding DINO: text-grounded detection
    dino_boxes = grounding_dino(frame, query)   # e.g., [{"box": [x,y,w,h], "score": 0.87}]
    
    # VLM: structured description
    vlm_output = vlm_describe(frame, f"Does this image contain {query}? Answer yes/no and explain.")
    
    # Consensus: at least 2 of 3 must agree
    votes = [
        clip_score > 0.25,
        len(dino_boxes) > 0,
        "yes" in vlm_output.lower(),
    ]
    
    return {
        "label": query,
        "confident": sum(votes) >= 2,
        "clip_score": clip_score,
        "dino_detections": len(dino_boxes),
        "vlm_agrees": "yes" in vlm_output.lower(),
    }
```

### Few-Shot Adapters on Frozen Embeddings

For domain-specific labels that foundation models miss (e.g., "this is a scene transition artifact" or "this video has a watermark at the bottom-right"), train lightweight classifiers:

1. Compute embeddings for the entire dataset (CLIP, SigLIP) — store in the lakehouse
2. Manually label **10-30 examples** per class
3. Train a linear probe or small MLP on the frozen embeddings
4. Apply to the entire dataset — runs in seconds since embeddings are pre-computed

This replaces traditional annotation campaigns (weeks, thousands of labels) with a few-shot approach (hours, dozens of labels).

### Why this matters

Auto-labeling with foundation models lets a small data team annotate billions of samples at a fraction of the cost of human annotation. The quality bar is set by the **consensus threshold** — strict consensus = fewer but more reliable labels, loose consensus = more labels with more noise. The right tradeoff depends on the training stage (loose for pre-training, strict for fine-tuning).

## Checkpoint

### Key Takeaways

**Re-captioning ROI**: Replacing alt-text with VLM captions is consistently the highest-impact data intervention. Do this before any model architecture changes — the quality gap is that large.

**Structured captions** ensure consistent coverage of scene, subjects, actions, style, and camera angle. This enables reliable conditioning at inference time and prevents models from learning inconsistent associations.

**Caption quality scoring** prevents garbage-in from VLM hallucinations. Score on length, specificity, field coverage, and repetition. Anything below threshold gets flagged for re-processing.

**Data mixing is underappreciated** — wrong ratios can tank model quality even with perfect data. The default (proportional by size) is almost always wrong. Use sqrt or quality-weighted strategies, and consider curriculum scheduling.

**Format choice** depends on your access pattern:
- Streaming-only → **WebDataset**
- Random access + large scale → **MDS**
- Video + random access → **Lance**
- Vector search needed → **Lance**

**Scale context**: production re-captioning runs on billions of images across hundreds of GPUs. Distributed inference is non-trivial (see notebook 19 on distributed training infrastructure).

### Further Reading

- Open-Sora 2.0 data pipeline: [arXiv 2503.09642](https://arxiv.org/abs/2503.09642) — detailed treatment of captioning at scale and data mixing for video generation
- HunyuanVideo: [arXiv 2412.03603](https://arxiv.org/abs/2412.03603) — structured caption format used in production video generation
- Notebook 09: Lance format deep dive
- Notebook 18: Caption quality filtering at scale
- Notebook 19: Distributed data pipeline infrastructure
